# NB_hospital_daily_summary — Gold: Hospital Daily Summary

Builds `adb_dev_healthcare.gold.hospital_daily_summary` — **one row per hospital per calendar day** — aggregating lab results with hospital and calendar dimensions, plus day-over-day trend via LAG.

**Load Type:** Full rebuild (CREATE OR REPLACE TABLE AS SELECT)

In [0]:
%sql
CREATE OR REPLACE TABLE adb_dev_healthcare.gold.hospital_daily_summary AS

WITH base_agg AS (
  SELECT
    lr.hospital_id,
    lr.test_date                                                       AS summary_date,
    COUNT(*)                                                           AS total_tests,
    COUNT(DISTINCT lr.patient_id)                                      AS unique_patients_tested,
    COUNT(DISTINCT lr.doctor_id)                                       AS active_doctor_count,
    COUNT(CASE WHEN lr.result_status = 'Abnormal' THEN 1 END)         AS abnormal_test_count,
    COUNT(CASE WHEN lr.result_status = 'Critical' THEN 1 END)         AS critical_test_count
  FROM adb_dev_healthcare.silver.lab_results lr
  GROUP BY lr.hospital_id, lr.test_date
),

enriched AS (
  SELECT
    ba.hospital_id,
    h.hospital_name,
    ba.summary_date,
    c.day_name,
    c.is_weekend,
    ba.total_tests,
    ba.unique_patients_tested,
    ba.active_doctor_count,
    ba.abnormal_test_count,
    ba.critical_test_count,
    ROUND(ba.abnormal_test_count / ba.total_tests * 100, 2)            AS abnormal_rate_pct,
    ROUND(ba.critical_test_count / ba.total_tests * 100, 2)            AS critical_rate_pct
  FROM      base_agg                                    ba
  LEFT JOIN adb_dev_healthcare.silver.hospitals           h  ON ba.hospital_id = h.hospital_id
  LEFT JOIN adb_dev_healthcare.silver.calendar            c  ON c.date_key     = CAST(DATE_FORMAT(ba.summary_date, 'yyyyMMdd') AS INT)
)

SELECT
  hospital_id,
  hospital_name,
  summary_date,
  day_name,
  is_weekend,
  total_tests,
  unique_patients_tested,
  active_doctor_count,
  abnormal_test_count,
  critical_test_count,
  abnormal_rate_pct,
  critical_rate_pct,
  LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date)  AS prev_day_total_tests,
  ROUND(
    CASE
      WHEN LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date) IS NULL
        OR LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date) = 0
      THEN NULL
      ELSE (total_tests - LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date))
           / LAG(total_tests) OVER (PARTITION BY hospital_id ORDER BY summary_date) * 100
    END, 2
  )                                                                      AS test_volume_change_pct
FROM enriched;

In [0]:
updated_df = spark.table("adb_dev_healthcare.gold.hospital_daily_summary")
records_count = updated_df.count()
dbutils.notebook.exit(str(records_count))